***Loading Datasets***

In [ ]:
import pandas as pd

df = pd.read_csv('2021.csv')
df.drop('Unnamed: 1',axis=1,inplace=True)       # removing index column
df.rename(columns={'Unnamed: 0':'Stage'},inplace=True)  # Required for prediction
df_c1 = df      # for Prediction
df1 = df.iloc[:16].reset_index(drop=True).fillna('')
df2 = df.iloc[16:32].reset_index(drop=True).fillna('')
df3 = df.iloc[32:].reset_index(drop=True).fillna('')
df_21 = pd.concat([df1,df2,df3],keys=['Opening Stage','Elimination Stage','Playoff Stage']).drop('Stage',axis=1)  # for other features... removing Stage column

df = pd.read_csv('2022.csv')
df.drop('Unnamed: 1',axis=1,inplace=True)
df.rename(columns={'Unnamed: 0':'Stage'},inplace=True)
df_c2 = df
df1 = df.iloc[:16].reset_index(drop=True).fillna('')
df2 = df.iloc[16:32].reset_index(drop=True).fillna('')
df3 = df.iloc[32:].reset_index(drop=True).fillna('')
df_22 = pd.concat([df1,df2,df3],keys=['Opening Stage','Elimination Stage','Playoff Stage']).drop('Stage',axis=1)

df = pd.read_csv('2023.csv')
df.drop('Unnamed: 1',axis=1,inplace=True)
df.rename(columns={'Unnamed: 0':'Stage'},inplace=True)
df_c3 = df
df1 = df.iloc[:16].reset_index(drop=True).fillna('')
df2 = df.iloc[16:32].reset_index(drop=True).fillna('')
df3 = df.iloc[32:].reset_index(drop=True).fillna('')
df_23 = pd.concat([df1,df2,df3],keys=['Opening Stage','Elimination Stage','Playoff Stage']).drop('Stage',axis=1)

df = pd.read_csv('2024.csv')
df.drop('Unnamed: 1',axis=1,inplace=True)
df.rename(columns={'Unnamed: 0':'Stage'},inplace=True)
df_c4 = df
df1 = df.iloc[:16].reset_index(drop=True).fillna('')
df2 = df.iloc[16:32].reset_index(drop=True).fillna('')
df3 = df.iloc[32:].reset_index(drop=True).fillna('')
df_24 = pd.concat([df1,df2,df3],keys=['Opening Stage','Elimination Stage','Playoff Stage']).drop('Stage',axis=1)


df = pd.concat([df_c1,df_c2,df_c3,df_c4]).reset_index(drop=True).fillna("")

***Prediction***

In [2]:
def calculate_avg_metrics(group):
    total_wins = group['Matches'].apply(lambda x: int(x.split('-')[0])).sum()
    total_losses = group['Matches'].apply(lambda x: int(x.split('-')[1])).sum()
    total_matches = total_wins + total_losses
    avg_win_rate = total_wins / total_matches
    avg_rd = group['RD'].mean()
    stages = group['Stage'].tolist()
    playoff_appearances = stages.count('Playoff Stage')
    return pd.Series({
        'Avg Win Rate': avg_win_rate,
        'Avg RD': avg_rd,
        'Playoff Appearances': playoff_appearances
    })

team_stats = df.groupby('Team',group_keys=False).apply(calculate_avg_metrics).reset_index()
team_stats

C:\Users\This PC\AppData\Local\Temp\ipykernel_3676\2059011531.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  team_stats = df.groupby('Team',group_keys=False).apply(calculate_avg_metrics).reset_index()


,Team,Avg Win Rate,Avg RD,Playoff Appearances
0,9INE,0.000000,-24.000000,0.0
1,9z Team,0.000000,-21.000000,0.0
2,AMKAL ESPORTS,0.000000,-17.000000,0.0
3,Apeks,0.533333,-1.500000,1.0
4,Astralis,0.428571,-4.666667,0.0
5,BIG,0.333333,-10.000000,0.0
6,Bad News Eagles,0.333333,-8.666667,0.0
7,CPH Flames,0.000000,-10.000000,1.0
8,Cloud9,0.583333,1.500000,1.0
9,Complexity Gaming,0.307692,-30.000000,0.0


In [3]:
def predict_outcome(team_stats, playoff_threshold=2, win_rate_threshold=0.4):
    likely_qualifiers = team_stats[
        (team_stats['Playoff Appearances'] >= playoff_threshold) &
        (team_stats['Avg Win Rate'] >= win_rate_threshold)
    ]
    
    likely_winners = likely_qualifiers.sort_values(by=['Avg RD', 'Avg Win Rate'], ascending=False).head(1)
    
    return likely_qualifiers['Team'], likely_winners['Team']

qualifiers, winners = predict_outcome(team_stats)

print("Teams likely to qualify:", qualifiers.tolist())
print("Teams likely to win:", winners.tolist())


Teams likely to qualify: ['FURIA Esports', 'FaZe Clan', 'G2 Esports', 'Heroic', 'Natus Vincere', 'Ninjas in Pyjamas', 'Team Spirit', 'Team Vitality']
Teams likely to win: ['G2 Esports']


***Win Rate***

In [4]:
def calculate_win_rate(row):
    wins = int(row['Matches'].split('-')[0])
    losses = int(row['Matches'].split('-')[1])
    total_matches = wins + losses
    win_rate = wins / total_matches
    return win_rate

# df_24['Win Rate'] = df_24.apply(calculate_win_rate, axis=1)
# df_24[['Team', 'Win Rate']]

df_24['Win Rate'] = df_24.apply(calculate_win_rate, axis=1)
# df_24['Win Rate'] = df_24['Win Rate'] * 100
# df_24['Win Rate'] = df_24['Win Rate'].apply(lambda x: f"{x:.0f}%")
df_24[['Team', 'Win Rate']]

Team  Win Rate
Opening Stage     0               HEROIC  1.000000
                  1               Cloud9  1.000000
                  2         Eternal Fire  0.750000
                  3             ECSTATIC  0.750000
                  4          paiN Gaming  0.750000
                  5     Imperial Esports  0.600000
                  6          The MongolZ  0.600000
                  7        FURIA Esports  0.600000
                  8                  SAW  0.400000
                  9               Legacy  0.400000
                  10         GamerLegion  0.400000
                  11  Lynn Vision Gaming  0.250000
                  12                ENCE  0.250000
                  13               Apeks  0.250000
                  14       AMKAL ESPORTS  0.000000
                  15                 KOI  0.000000
Elimination Stage 0          Team Spirit  1.000000
                  1                 MOUZ  1.000000
                  2         Eternal Fire  0.750000
                  3               Cloud9  0.750000
                  4        Team Vitality  0.750000
                  5        Natus Vincere  0.600000
                  6           G2 Esports  0.600000
                  7            FaZe Clan  0.600000
                  8    Complexity Gaming  0.400000
                  9           Virtus.pro  0.400000
                  10         paiN Gaming  0.400000
                  11    Imperial Esports  0.250000
                  12            ECSTATIC  0.250000
                  13              HEROIC  0.250000
                  14         The MongolZ  0.000000
                  15       FURIA Esports  0.000000
Playoff Stage     0        Natus Vincere  1.000000
                  1            FaZe Clan  0.666667
                  2        Team Vitality  0.500000
                  3           G2 Esports  0.500000
                  4          Team Spirit  0.000000
                  5               Cloud9  0.000000
                  6         Eternal Fire  0.000000
                  7                 MOUZ  0.000000

***Round Difference (RD) Analysis***

In [ ]:
# Win Rate Required
correlation = df_24['RD'].corr(df_24['Win Rate'])
print(f"Correlation between RD and Win Rate: {correlation:.2f}")

Correlation between RD and Win Rate: 0.79


***Average Rounds per Game***

In [6]:
def total_matches(row):
    wins = int(row['Matches'].split('-')[0])
    losses = int(row['Matches'].split('-')[1])
    return wins + losses

df_24['Total Matches'] = df_24.apply(total_matches, axis=1)
df_24['Rounds Won'] = df_24['Rounds'].str.split('-').str[0].astype(int)
df_24['Rounds Lost'] = df_24['Rounds'].str.split('-').str[1].astype(int)

df_24['Avg Rounds Won'] = (df_24['Rounds Won'] / df_24['Total Matches']).astype(int)
df_24['Avg Rounds Lost'] = (df_24['Rounds Lost'] / df_24['Total Matches']).astype(int)

df_24[['Team', 'Avg Rounds Won', 'Avg Rounds Lost']]

Team  Avg Rounds Won  Avg Rounds Lost
Opening Stage     0               HEROIC              19               13
                  1               Cloud9              19               15
                  2         Eternal Fire              19               15
                  3             ECSTATIC              18               14
                  4          paiN Gaming              13               10
                  5     Imperial Esports              22               21
                  6          The MongolZ              20               16
                  7        FURIA Esports              20               15
                  8                  SAW              16               19
                  9               Legacy              14               15
                  10         GamerLegion              17               18
                  11  Lynn Vision Gaming              10               15
                  12                ENCE              13               18
                  13               Apeks              11               16
                  14       AMKAL ESPORTS              15               21
                  15                 KOI              10               18
Elimination Stage 0          Team Spirit              21               13
                  1                 MOUZ              17                8
                  2         Eternal Fire              17               12
                  3               Cloud9              18               12
                  4        Team Vitality              18               17
                  5        Natus Vincere              21               23
                  6           G2 Esports              22               18
                  7            FaZe Clan              16               14
                  8    Complexity Gaming              17               23
                  9           Virtus.pro              18               20
                  10         paiN Gaming              21               21
                  11    Imperial Esports              13               16
                  12            ECSTATIC              20               23
                  13              HEROIC              12               17
                  14         The MongolZ              20               23
                  15       FURIA Esports              13               22
Playoff Stage     0        Natus Vincere              31               28
                  1            FaZe Clan              33               32
                  2        Team Vitality              27               20
                  3           G2 Esports              31               25
                  4          Team Spirit              40               45
                  5               Cloud9              10               26
                  6         Eternal Fire              22               29
                  7                 MOUZ              15               26

***Ranking Teams by Round Efficiency***

In [7]:
df_24['Total Rounds Played'] = df_24['Rounds Won'] + df_24['Rounds Lost']
df_24['Round Efficiency'] = df_24['Rounds Won'] / df_24['Total Rounds Played']

df_24[['Team', 'Round Efficiency']]

Team  Round Efficiency
Opening Stage     0               HEROIC          0.591837
                  1               Cloud9          0.557692
                  2         Eternal Fire          0.550000
                  3             ECSTATIC          0.558140
                  4          paiN Gaming          0.568421
                  5     Imperial Esports          0.518349
                  6          The MongolZ          0.560440
                  7        FURIA Esports          0.571429
                  8                  SAW          0.461111
                  9               Legacy          0.473333
                  10         GamerLegion          0.486188
                  11  Lynn Vision Gaming          0.396040
                  12                ENCE          0.429688
                  13               Apeks          0.409091
                  14       AMKAL ESPORTS          0.422018
                  15                 KOI          0.360465
Elimination Stage 0          Team Spirit          0.613208
                  1                 MOUZ          0.675325
                  2         Eternal Fire          0.579832
                  3               Cloud9          0.598361
                  4        Team Vitality          0.517241
                  5        Natus Vincere          0.475771
                  6           G2 Esports          0.550000
                  7            FaZe Clan          0.531646
                  8    Complexity Gaming          0.423645
                  9           Virtus.pro          0.476440
                  10         paiN Gaming          0.495370
                  11    Imperial Esports          0.445378
                  12            ECSTATIC          0.468571
                  13              HEROIC          0.428571
                  14         The MongolZ          0.465649
                  15       FURIA Esports          0.377358
Playoff Stage     0        Natus Vincere          0.525424
                  1            FaZe Clan          0.512690
                  2        Team Vitality          0.568421
                  3           G2 Esports          0.552632
                  4          Team Spirit          0.470588
                  5               Cloud9          0.277778
                  6         Eternal Fire          0.431373
                  7                 MOUZ          0.365854

***Identify Top 3 Teams by Win Rate***

In [9]:
top_3_teams = df_24.sort_values(by='Win Rate', ascending=False).head(3).reset_index()
print(top_3_teams[['Team', 'Win Rate']])


          Team  Win Rate
0       HEROIC       1.0
1       Cloud9       1.0
2  Team Spirit       1.0
